In [1]:
import numpy as np
import pandas as pd
from scipy import stats

gaps = pd.read_csv('data/teammate_gaps.csv')
quali = pd.read_csv('data/qualifying_clean.csv', parse_dates=['date'])
print(len(gaps), 'comparisons |', gaps['outlier'].sum(), 'flagged outliers')   # expect 2645 | 210

2655 comparisons | 210 flagged outliers


In [2]:
MIN_SESSIONS = 8
Z = stats.norm.ppf(0.975)

def wilson(k, n, z=Z):
    p = k / n
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centre - half, centre + half

def pair_season_stats(g):
    clean = g.loc[~g['outlier'], 'gap']
    n, m = len(clean), clean.mean()
    sd = clean.std(ddof=1)
    t_half = stats.t.ppf(0.975, n - 1) * sd / np.sqrt(n)
    sd_low = np.sqrt((n - 1) * sd**2 / stats.chi2.ppf(0.975, n - 1))
    sd_high = np.sqrt((n - 1) * sd**2 / stats.chi2.ppf(0.025, n - 1))
    wins, decided = (g['gap'] < 0).sum(), (g['gap'] != 0).sum()   # a dead heat counts for neither
    h_low, h_high = wilson(wins, decided)
    return pd.Series({'team': g['team'].iloc[0], 'n_sessions': len(g), 'n_outliers': g['outlier'].sum(),
                      'n_clean': n, 'mean_gap': m, 'mean_ci_low': m - t_half, 'mean_ci_high': m + t_half,
                      'median_gap': g['gap'].median(), 'sd_gap': sd, 'sd_ci_low': sd_low, 'sd_ci_high': sd_high,
                      'h2h_A_wins': wins, 'h2h_sessions': decided, 'h2h_A_prop': wins / decided,
                      'h2h_ci_low': h_low, 'h2h_ci_high': h_high})

counts = gaps.groupby(['A', 'B', 'year'])['gap'].transform('size')
pair_seasons = (gaps[counts >= MIN_SESSIONS]
                .groupby(['year', 'A', 'B']).apply(pair_season_stats, include_groups=False)
                .reset_index())
int_cols = ['n_sessions', 'n_outliers', 'n_clean', 'h2h_A_wins', 'h2h_sessions']
pair_seasons[int_cols] = pair_seasons[int_cols].astype(int)

print(len(pair_seasons), 'pair-seasons')   # expect 137
assert pair_seasons.notna().all().all(), 'empty cells in pair_seasons'
pair_seasons.round(3).head()

137 pair-seasons


,year,A,B,team,n_sessions,n_outliers,n_clean,mean_gap,mean_ci_low,mean_ci_high,median_gap,sd_gap,sd_ci_low,sd_ci_high,h2h_A_wins,h2h_sessions,h2h_A_prop,h2h_ci_low,h2h_ci_high
0,2014,adrian-sutil,esteban-gutierrez,sauber,18,0,18,-0.192,-0.549,0.166,-0.095,0.719,0.539,1.078,10,18,0.556,0.337,0.754
1,2014,daniel-ricciardo,sebastian-vettel,red-bull,18,2,16,-0.124,-0.345,0.097,-0.223,0.414,0.306,0.641,11,18,0.611,0.386,0.797
2,2014,daniil-kvyat,jean-eric-vergne,toro-rosso,19,1,18,-0.095,-0.319,0.130,-0.122,0.451,0.339,0.677,12,19,0.632,0.410,0.809
3,2014,felipe-massa,valtteri-bottas,williams,19,1,18,0.215,0.016,0.414,0.198,0.400,0.300,0.600,7,19,0.368,0.191,0.590
4,2014,fernando-alonso,kimi-raikkonen,ferrari,19,1,18,-0.475,-0.756,-0.194,-0.679,0.565,0.424,0.848,16,19,0.842,0.624,0.945


In [3]:
# Spot test: Russell-Antonelli 2025
pair_seasons[(pair_seasons['A'] == 'george-russell') & (pair_seasons['B'] == 'kimi-antonelli')
             & (pair_seasons['year'] == 2025)].round(3).T

,122
year,2025
A,george-russell
B,kimi-antonelli
team,mercedes
n_sessions,24
n_outliers,4
n_clean,20
mean_gap,-0.376
mean_ci_low,-0.483
mean_ci_high,-0.27


In [4]:
pair_seasons.to_csv('data/pair_seasons.csv', index=False)
print('saved', len(pair_seasons), 'rows to pair_seasons.csv')

saved 137 rows to pair_seasons.csv


In [5]:
def pilot_row(a, b, year):
    r = pair_seasons[(pair_seasons['A'] == a) & (pair_seasons['B'] == b) & (pair_seasons['year'] == year)].iloc[0]
    return r

rk = pilot_row('george-russell', 'kimi-antonelli', 2025)
rh = pilot_row('george-russell', 'lewis-hamilton', 2024)

se_rk, se_rh = rk['sd_gap'] / np.sqrt(rk['n_clean']), rh['sd_gap'] / np.sqrt(rh['n_clean'])
chain_mean = rh['mean_gap'] - rk['mean_gap']
chain_se = np.sqrt(se_rk**2 + se_rh**2)
chain_df = chain_se**4 / (se_rk**4 / (rk['n_clean'] - 1) + se_rh**4 / (rh['n_clean'] - 1))
chain_half = stats.t.ppf(0.975, chain_df) * chain_se
# exact version of the chain (ratio of ratios) to show the log approximation is harmless
chain_exact = 100 * ((1 + rh['mean_gap'] / 100) / (1 + rk['mean_gap'] / 100) - 1)

cols = ['n_sessions', 'n_outliers', 'mean_gap', 'mean_ci_low', 'mean_ci_high', 'median_gap', 'sd_gap',
        'h2h_A_wins', 'h2h_sessions', 'h2h_A_prop', 'h2h_ci_low', 'h2h_ci_high']
table3 = pd.DataFrame([rk[['A', 'B', 'year'] + cols], rh[['A', 'B', 'year'] + cols]])
table3.loc[len(table3)] = {'A': 'kimi-antonelli', 'B': 'lewis-hamilton', 'year': '2025 via 2024',
                           'mean_gap': chain_mean, 'mean_ci_low': chain_mean - chain_half,
                           'mean_ci_high': chain_mean + chain_half}
print(f'chain: {chain_mean:+.3f}% (exact ratio {chain_exact:+.3f}%), SE {chain_se:.3f}, df {chain_df:.1f}')
table3.round(3)

chain: +0.163% (exact ratio +0.164%), SE 0.076, df 38.8


,A,B,year,n_sessions,n_outliers,mean_gap,mean_ci_low,mean_ci_high,median_gap,sd_gap,h2h_A_wins,h2h_sessions,h2h_A_prop,h2h_ci_low,h2h_ci_high
122,george-russell,kimi-antonelli,2025,24.0,4.0,-0.376,-0.483,-0.270,-0.384,0.227,21.0,24.0,0.875,0.690,0.957
111,george-russell,lewis-hamilton,2024,24.0,3.0,-0.213,-0.330,-0.096,-0.224,0.258,19.0,24.0,0.792,0.595,0.908
2,kimi-antonelli,lewis-hamilton,2025 via 2024,NaN,NaN,0.163,0.010,0.317,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
table3.to_csv('data/table3_pilot.csv', index=False)
print('saved table3_pilot.csv')

saved table3_pilot.csv


In [7]:
import yaml
from pathlib import Path

F1DB = Path('../f1db-main/f1db-main/src/data')
LOADER = getattr(yaml, 'CSafeLoader', yaml.SafeLoader)
def load(path): return yaml.load(open(path, encoding='utf-8'), Loader=LOADER)

# team lineage: (constructorId, year) -> current name of that team
lineage = {}
for f in (F1DB / 'constructors').glob('*.yml'):
    chron = load(f).get('chronology') or []
    for e in chron:
        for y in range(e['yearFrom'], (e['yearTo'] or 2026) + 1):
            lineage[(e['constructorId'], y)] = chron[-1]['constructorId']
def team_line(cid, year): return lineage.get((cid, year), cid)

# every Grand Prix start since 1950, and the length of each season
NOT_STARTED = {'DNQ', 'DNPQ', 'DNS'}
starts, n_rounds = [], {}
for season in sorted((F1DB / 'seasons').iterdir(), key=lambda p: int(p.name)):
    year = int(season.name)
    race_dirs = sorted((season / 'races').iterdir())
    n_rounds[year] = len(race_dirs)
    for race_dir in race_dirs:
        results = race_dir / 'race-results.yml'
        if not results.exists():
            continue
        date = pd.Timestamp(load(race_dir / 'race.yml')['date'])
        starts += [{'date': date, 'year': year, 'driverId': r['driverId'], 'line': team_line(r['constructorId'], year)}
                   for r in load(results) if str(r['position']) not in NOT_STARTED]
starts = pd.DataFrame(starts)
print(len(starts), 'starts from', starts['year'].min(), 'to', starts['year'].max())

25905 starts from 1950 to 2026


In [8]:
start_dates = {d: np.sort(g['date'].to_numpy()) for d, g in starts.groupby('driverId')}
team_seasons = {k: np.unique(s.to_numpy()) for k, s in starts.groupby(['driverId', 'line'])['year']}

def prior_starts(driver, date):
    return int(np.searchsorted(start_dates.get(driver, np.array([], 'datetime64[ns]')), np.datetime64(date)))

def prior_team_years(driver, line, year):
    yrs = team_seasons.get((driver, line), np.array([]))
    return int((yrs < year).sum())

race_info = quali.drop_duplicates(['year', 'round']).set_index(['year', 'round'])[['date', 'circuit', 'circuitType']]
age = quali.set_index(['year', 'round', 'driverId'])['age']

feat = gaps[['year', 'round', 'team', 'A', 'B', 'segment', 'gap', 'outlier']].copy()
feat = feat.join(race_info, on=['year', 'round']).rename(columns={'circuitType': 'circuit_type'})
feat['round_frac'] = feat['round'] / feat['year'].map(n_rounds)
feat['reg_change'] = feat['year'].isin([2014, 2017, 2022, 2026]).astype(int)
for side in ['A', 'B']:
    feat[f'age_{side}'] = [age[(y, r, d)] for y, r, d in zip(feat['year'], feat['round'], feat[side])]
    feat[f'starts_{side}'] = [prior_starts(d, dt) for d, dt in zip(feat[side], feat['date'])]
    feat[f'team_years_{side}'] = [prior_team_years(d, team_line(t, y), y)
                                  for d, t, y in zip(feat[side], feat['team'], feat['year'])]
feat['age_diff'] = feat['age_A'] - feat['age_B']
feat['exp_diff'] = feat['starts_A'] - feat['starts_B']
feat['team_years_diff'] = feat['team_years_A'] - feat['team_years_B']
feat = feat.drop(columns='date')

assert feat.notna().all().all() and len(feat) == len(gaps)
feat.describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
year,2655.0,2020.12,3.73,2014.00,2017.00,2020.00,2023.00,2026.00
round,2655.0,10.97,6.14,1.00,6.00,11.00,16.00,24.00
gap,2655.0,-0.06,2.02,-48.73,-0.42,-0.07,0.29,43.91
round_frac,2655.0,0.52,0.29,0.04,0.27,0.52,0.76,1.00
reg_change,2655.0,0.30,0.46,0.00,0.00,0.00,1.00,1.00
age_A,2655.0,27.85,5.33,18.58,24.05,26.82,30.46,45.20
starts_A,2655.0,121.94,94.66,0.00,44.00,104.00,179.00,442.00
team_years_A,2655.0,2.26,2.21,0.00,0.00,2.00,4.00,9.00
age_B,2655.0,28.13,5.82,17.45,23.47,27.03,31.95,42.15
starts_B,2655.0,125.01,99.30,0.00,37.00,104.00,193.00,396.00


In [9]:
# Spot checks: a rookie's first race, and a long-serving driver
first = feat[(feat['year'] == 2025) & (feat['round'] == 1) & (feat['team'] == 'mercedes')]
print(first[['A', 'B', 'starts_A', 'starts_B', 'team_years_A', 'team_years_B', 'age_diff', 'round_frac']].to_string(index=False))
# expect Antonelli (B) 0 starts and 0 team years; Russell's team years include his one-off 2020 Sakhir drive
print(feat.groupby('circuit_type').size(), feat.groupby('reg_change').size(), sep='\n')

             A              B  starts_A  starts_B  team_years_A  team_years_B  age_diff  round_frac
george-russell kimi-antonelli       128         0             4             0   8.52293    0.041667
circuit_type
RACE      1906
ROAD        78
STREET     671
dtype: int64
reg_change
0    1870
1     785
dtype: int64


In [10]:
feat.to_csv('data/features.csv', index=False)
print('saved', len(feat), 'rows and', feat.shape[1], 'columns to features.csv')

saved 2655 rows and 21 columns to features.csv
